In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import torch
import cv2
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from tqdm import tqdm  # progress bar

device = "cuda" if torch.cuda.is_available() else "cpu"
# Load BLIP2
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", device_map=device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def get_video_paths_and_labels(video_root):
    video_paths = []
    video_labels = []  # simpan nama folder (emosi)
    for emotion in os.listdir(video_root):
        emotion_dir = os.path.join(video_root, emotion)
        if os.path.isdir(emotion_dir):
            for f in os.listdir(emotion_dir):
                if f.endswith(".mp4"):  # asumsi format .mp4
                    video_paths.append(os.path.join(emotion_dir, f))
                    video_labels.append(emotion)

    print(f"Total videos found: {len(video_paths)}")
    return video_paths, video_labels

def generate_captions(video_path, n_frames=8):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(frame_count // n_frames, 1)

    captions = []
    for i in range(0, frame_count, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            continue
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        inputs = processor(images=image, return_tensors="pt").to(model.device)
        with torch.no_grad():
            generated_ids = model.generate(**inputs)
            caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        captions.append(caption)

    cap.release()

    if len(captions) == 0:
        return "EMPTY"
    else:
        return " ".join(captions)  # combine captions from sampled frames

def generate_captions_all(video_paths, n_frames=8):
    captions = []
    for path in tqdm(video_paths, total=len(video_paths)):
        caption = generate_captions(path, n_frames)
        captions.append(caption)
    return captions

In [ ]:
video_root = "/content/drive/MyDrive/Satria_Data_2025/semifinal/videos/"
video_paths, video_labels = get_video_paths_and_labels(video_root)

Total videos found: 998


In [ ]:
captions = generate_captions_all(video_paths)
result = pd.DataFrame({'path': video_paths, 'caption': captions})

100%|██████████| 998/998 [2:51:12<00:00, 10.29s/it]


In [ ]:
data = result

# Translate

In [ ]:
import ast
if isinstance(data['caption'].iloc[0], str):
    data['caption'] = data['caption'].apply(ast.literal_eval)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.notebook import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

# Replace with a newer/better model you prefer
# model_name = "Helsinki-NLP/opus-mt-en-id"
model_name = "facebook/nllb-200-1.3B"
# Or if NusaMT-7B is available for en→id, use that

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def translate_batch(sentences, batch_size=16, src_lang="eng_Latn", tgt_lang="ind_Latn"):
    results = []

    tokenizer.src_lang = src_lang

    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i+batch_size]

        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        # Get the language ID safely
        tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
        if tgt_lang_id is None:
            tgt_lang_id = tokenizer.convert_tokens_to_ids(tokenizer.bos_token)

        # Generate translations
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tgt_lang_id,
            num_beams=4,
            max_length=256
        )

        translations = [tokenizer.decode(t, skip_special_tokens=True) for t in outputs]
        del inputs, outputs
        gc.collect()
        torch.cuda.empty_cache()
        results.extend(translations)

    return results

In [ ]:
# Flatten all captions
all_captions = [cap for caps in data['caption'] for cap in caps]

# Translate all
translated_captions = translate_batch(all_captions, batch_size=16)

# Reconstruct same shape
translated_results = []
idx = 0
for caps in data['caption']:
    translated_video = translated_captions[idx : idx + len(caps)]
    translated_results.append(translated_video)
    idx += len(caps)

# Add to DataFrame
data['caption_id'] = translated_results

# # Save
df.to_csv("captions_translated.csv", index=False)
print("Translation complete! Saved to captions_translated.csv")